# EVALUATION NOTEBOOK TO MAKE GRAPHS FOR DISSERTATION

All graphs in evaluation (and appendices) made here.

UNUSED CODE COMMENTED OUT

## Setup

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd

url = ("https://github.com/google/fonts/raw/main/ofl/ebgaramond/"
       "EBGaramond%5Bwght%5D.ttf")
cache_dir = Path.home() / ".cache" / "matplotlib_fonts"
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / "EBGaramond.ttf"

req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=15) as r:
    target.write_bytes(r.read())

fm.fontManager.addfont(str(target))
target_resolved = str(target.resolve())
name = next(f.name for f in fm.fontManager.ttflist
            if str(Path(f.fname).resolve()) == target_resolved)
plt.rcParams["font.family"] = name
print(f"Downloaded and registered '{name}' from Google Fonts.")

plt.rcParams.update({
    "font.family":       "EB Garamond",
    "font.size":         20,
    "axes.titlesize":    20,
    "axes.labelsize":    20,
    "xtick.labelsize":   14,
    "ytick.labelsize":   14,
    "legend.fontsize":   14,
    "mathtext.fontset":  "stix",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.25,
    "grid.linewidth":    0.6,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox":      "tight",
})

# constants
RESULTS_DIR = Path("results")
NOISE_LEVELS = {0.0: "Clean", 0.025: "Med (2.5%)", 0.05: "High (5%)"}
ABLATION_NOISE_LEVELS = {0.0: "Clean (0%)", 0.025: "Noisy (2.5%)"}
METRICS = [ ("success_rate", "Success rate"), ("r2_mean", r"Mean $R^2$"), ("time_mean", "Mean time (s)"),
]

PALETTE = {0.0: "#298c8c", 0.025: "#f1a226", 0.05: "#3D5A80"}
COLOR_DEFAULT_LINE = "#888888"
 
ERROR_KIND = "sem"

_ERR_COLS = {
    "std": {"r2_mean": "r2_std", "time_mean": "time_std", "time_mean_found": "time_std_found"},
    "sem": {"r2_mean": "r2_sem", "time_mean": "time_sem", "time_mean_found": "time_sem_found"},
}


In [ ]:
from symbolic_discovery.utils.analysis import load_csvs, aggregate, parse_variants

PRETTY = {"bacon3f": "BACON.3F", "bacon7f": "BACON.7F", "pysr": "PySR", "pysr_norm": "PySR (normalised)", 
          "baseline": "Baseline", "no_early_stop": "No early stop", "no_uncorr": "No uncorr.", "no_relax": "No relaxation", "no_folds": "No folds"}

def _yerr(frame: pd.DataFrame, mean_col: str):
    col = _ERR_COLS.get(ERROR_KIND, _ERR_COLS["sem"]).get(mean_col)
    if col is None or col not in frame.columns:
        return None
    return frame[col].fillna(0).values

def plot_validation_results(agg, models_order, *, title_prefix="", metrics=METRICS, noise_levels=NOISE_LEVELS, palette=PALETTE, ymargin=0.10):
    subs = {n: (agg[(agg["variant"].isin(models_order)) & (agg["noise"] == n)]
                .set_index("variant").reindex(models_order))
            for n in noise_levels}
            
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4.5), constrained_layout=True)
    if len(metrics) == 1: 
        axes = [axes]

    plt.rcParams["ytick.labelsize"] = 14
    plt.rcParams["xtick.labelsize"] = 14
    
    n_noise = len(noise_levels)
    x_indices = np.arange(len(models_order))
    bar_width = 0.9 / n_noise
    offsets = (np.arange(n_noise) - (n_noise - 1) / 2) * bar_width
    bar_colors = [palette.get(n, "#666666") for n in noise_levels]
    
    for ax, (col, ylabel) in zip(axes, metrics):
        lo_pts, hi_pts = [], []
        
        for i, (n, lbl) in enumerate(noise_levels.items()):
            vals = subs[n][col].values
            yerr = _yerr(subs[n], col)
            
            bars = ax.bar(x_indices + offsets[i], vals, bar_width, yerr=yerr, capsize=3, error_kw={"linewidth": 1.6, "ecolor": "#333333"}, color=bar_colors[i], edgecolor="none", label=lbl)
                        
            ax.bar_label(bars, fmt="%.3f", padding=3, color="#444444")
            
            err_arr = yerr if yerr is not None else np.zeros_like(vals)
            lo_pts.append(np.asarray(vals, float) - np.asarray(err_arr, float))
            hi_pts.append(np.asarray(vals, float) + np.asarray(err_arr, float))
            
        ax.set_xticks(x_indices)
        ax.set_xticklabels([PRETTY.get(m, m) for m in models_order], rotation=0, ha="center", fontsize=20)
        
        ax.set_title(ylabel, loc="left", pad=10)
        ax.tick_params(axis="y")
        
        lo_arr = pd.Series(np.concatenate(lo_pts)).dropna().values
        hi_arr = pd.Series(np.concatenate(hi_pts)).dropna().values
        if len(lo_arr) and len(hi_arr):
            lo, hi = float(lo_arr.min()), float(hi_arr.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi) * 0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.10))

    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(
        handles, labels, 
        loc="lower left",
        bbox_to_anchor=(0.0, 1.1),
        ncol=2,
        frameon=False, 
    )
    
    '''fig.suptitle(
        f"{title_prefix}Validation Results" + f' (error bars: ±1 {"SEM" if ERROR_KIND == "sem" else "SD"})', 
        y=1.15
    )'''
                
    # fig.tight_layout()
    fig.savefig(Path("figures/validation.png"))
    plt.show()

def plot_ablation(agg, variants_order, *, title_prefix="", metrics=METRICS, noise_levels=ABLATION_NOISE_LEVELS, palette=PALETTE, ymargin=0.10):
    subs = {n: (agg[(agg["variant"].isin(variants_order)) & (agg["noise"] == n)]
                .set_index("variant").reindex(variants_order))
            for n in noise_levels}
    if all(s.empty or s[[c for c, _ in metrics]].isna().all().all()
           for s in subs.values()):
        return
        
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4.5), constrained_layout=True)
    if len(metrics) == 1: 
        axes = [axes]

    plt.rcParams["ytick.labelsize"] = 14
    plt.rcParams["xtick.labelsize"] = 14
 
    n_noise = len(noise_levels)
    x = np.arange(len(variants_order))
    bw = 0.9 / n_noise
    offsets = (np.arange(n_noise) - (n_noise - 1)/2) * bw
    bar_colors = [palette.get(n, "#666") for n in noise_levels]
 
    for ax, (col, ylabel) in zip(axes, metrics):
        lo_pts, hi_pts = [], []
        for i, (n, lbl) in enumerate(noise_levels.items()):
            vals = subs[n][col].values
            yerr = _yerr(subs[n], col)
            bars = ax.bar(x + offsets[i], vals, bw,
                          yerr=yerr, capsize=3,
                          error_kw={"linewidth": 1.6, "ecolor": "#333"},
                          color=bar_colors[i], edgecolor="none", label=lbl)
            ax.bar_label(bars, fmt="%.3f", padding=3, color="#444")
            err = yerr if yerr is not None else np.zeros_like(vals)
            lo_pts.append(np.asarray(vals, float) - np.asarray(err, float))
            hi_pts.append(np.asarray(vals, float) + np.asarray(err, float))
 
        ax.set_xticks(x)
        ax.set_xticklabels([PRETTY.get(m, m) for m in variants_order], rotation=0, ha="center", fontsize=20)
        ax.set_title(ylabel, loc="left", pad=10)
        ax.tick_params(axis="y")
 
        lo_arr = pd.Series(np.concatenate(lo_pts)).dropna().values
        hi_arr = pd.Series(np.concatenate(hi_pts)).dropna().values
        if len(lo_arr) and len(hi_arr):
            lo, hi = float(lo_arr.min()), float(hi_arr.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi)*0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span*ymargin, hi + span*(ymargin + 0.10))

    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(
        handles, labels, 
        loc="lower left",
        bbox_to_anchor=(0.0, 1.1), 
        ncol=2,
        frameon=False, 
    )
 
    # fig.suptitle(f"{title_prefix} — Ablation across noise levels" + f' (error bars: ±1 {"SEM" if ERROR_KIND == "sem" else "SD"})')
    fig.savefig(Path(f"figures/{title_prefix}.png"))
    plt.show()
 
 
def plot_sensitivity(agg, param_name, dtype, *, title_prefix="", metrics=METRICS, default_value=None, noise_levels=NOISE_LEVELS, palette=PALETTE):
    sub = agg[agg["param"] == param_name].copy()
    if sub.empty:
        print(f"  (no data for {param_name})")
        return
 
    sub["param_value"] = pd.to_numeric(sub["param_value"])
    sub = sub.sort_values("param_value")
 
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4.5), constrained_layout=True)
    if len(metrics) == 1: 
        axes = [axes]

    plt.rcParams["ytick.labelsize"] = 20
    plt.rcParams["xtick.labelsize"] = 20
 
    for noise_val, label in noise_levels.items():
        cell = sub[sub["noise"] == noise_val]
        if cell.empty: 
            continue
        color = palette[noise_val]
        for ax, (col, _) in zip(axes, metrics):
            yerr = _yerr(cell, col)
            ax.errorbar(cell["param_value"], cell[col], yerr=yerr,
                        marker="o", color=color,
                        linewidth=1.6, markersize=5.5,
                        capsize=3, elinewidth=0.8, label=label)
 
    for ax, (col, ylabel) in zip(axes, metrics):
        ax.set_xlabel(param_name)
        ax.set_title(ylabel, loc="left", pad=10)
        ax.tick_params(axis="y")
 
        if default_value is not None:
            ax.axvline(default_value, color=COLOR_DEFAULT_LINE,
                       linestyle=":", linewidth=1.8, zorder=0)
 
        if col == "success_rate":
            ax.set_ylim(0, 1.05)
        elif col == "r2_mean":
            vals = sub[col].dropna()
            if len(vals) and (vals.max() - vals.min()) < 0.05:
                ax.set_ylim(max(0.0, vals.min() - 0.005),
                            min(1.001, vals.max() + 0.003))
                
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(
        handles, labels, 
        loc="lower left",
        bbox_to_anchor=(0.0, 1.1), 
        ncol=2,
        frameon=False, 
    )
 
    # fig.suptitle(f"{title_prefix} Sensitivity: {param_name}" + f' (error bars: ±1 {"SEM" if ERROR_KIND == "sem" else "SD"})')
    fig.savefig(Path(f"figures/{title_prefix}_{param_name}.png"))
    plt.show()

def plot_sensitivity_grid(
    agg, param_map, *,
    title_prefix="",
    metrics=METRICS,
    default_values=None,
    noise_levels=NOISE_LEVELS,
    palette=PALETTE,
):
    params = []
    for _short, (full_name, dtype) in param_map.items():
        if not agg[agg["param"] == full_name].empty:
            params.append((full_name, dtype))
    if not params:
        print("  (no data)")
        return

    nrows, ncols = len(params), len(metrics)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5 * ncols, 3.5 * nrows),
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)
    if ncols == 1:
        axes = axes.reshape(-1, 1)

    plt.rcParams["ytick.labelsize"] = 20
    plt.rcParams["xtick.labelsize"] = 20

    for r, (param_name, _dtype) in enumerate(params):
        sub = agg[agg["param"] == param_name].copy()
        sub["param_value"] = pd.to_numeric(sub["param_value"])
        sub = sub.sort_values("param_value")
        default_value = default_values.get(param_name) if default_values else None

        for noise_val, label in noise_levels.items():
            cell = sub[sub["noise"] == noise_val]
            if cell.empty:
                continue
            color = palette[noise_val]
            for c, (col, _) in enumerate(metrics):
                ax = axes[r, c]
                yerr = _yerr(cell, col)
                ax.errorbar(
                    cell["param_value"], cell[col], yerr=yerr,
                    marker="o", color=color,
                    linewidth=1.6, markersize=5.5,
                    capsize=3, elinewidth=0.8, label=label,
                )

        for c, (col, ylabel) in enumerate(metrics):
            ax = axes[r, c]
            ax.set_xlabel(param_name)
            ax.tick_params(axis="y")

            if r == 0:
                ax.set_title(ylabel, loc="left", pad=10)

            if default_value is not None:
                ax.axvline(
                    default_value, color=COLOR_DEFAULT_LINE,
                    linestyle=":", linewidth=1.8, zorder=0,
                )

            if col == "success_rate":
                ax.set_ylim(0, 1.05)
            elif col == "r2_mean":
                vals = sub[col].dropna()
                if len(vals) and (vals.max() - vals.min()) < 0.05:
                    ax.set_ylim(
                        max(0.0, vals.min() - 0.005),
                        min(1.001, vals.max() + 0.003),
                    )

    handles, labels = [], []
    for ax in axes.flat:
        h, lab = ax.get_legend_handles_labels()
        for hi, li in zip(h, lab):
            if li not in labels:
                handles.append(hi)
                labels.append(li)
    if handles:
        fig.legend(
            handles, labels,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.0),
            ncol=len(labels),
            frameon=False,
        )

    fig.savefig(Path(f"figures/{title_prefix}_sensitivity.png"))
    plt.show()

## 1 Validation: full experiment sweep with BACONs

In [ ]:
validation = load_csvs([RESULTS_DIR / "validation.csv"])

In [ ]:
if validation is not None:
    overall = aggregate(validation, ["variant", "noise"]).sort_values(
        "success_rate", ascending=False
    )
    display(overall)

    print("\nDatasets each (variant, noise) FAILED on:")
    for (variant, noise), _ in validation.groupby(["variant", "noise"], dropna=False):
        failed = (
            validation[
                (validation["variant"] == variant)
                & (validation["noise"] == noise)
                & ~validation["found"]
            ]["dataset"].unique().tolist()
        )
        print(f"  {variant} @ noise={noise}: {len(failed)} failures — {sorted(failed)}")
    
    plot_validation_results(overall, models_order=["bacon3f", "bacon7f"])


## 2 BACON.3F ablations

### 2.1 Early stop

In [ ]:
abl_early_stop = load_csvs([RESULTS_DIR / "ablation_early_stop.csv"])
ES_ABL_ORDER = ["baseline", "no_early_stop"]

In [ ]:
if abl_early_stop is not None:
    abl_early_stop_agg = aggregate(abl_early_stop, ["variant", "noise"])
    display(abl_early_stop_agg.sort_values(["noise", "variant"]))
    plot_ablation(abl_early_stop_agg, ES_ABL_ORDER, title_prefix="BACON.3F Early Stop")

### 2.2 Uncorrelatedness check

Run with depth 5; depth 6 takes an estimated several days
without the uncorrelatedness gate.

In [ ]:
abl_rgate = load_csvs([RESULTS_DIR / "ablation_uncorrelatedness.csv"])
B3F_RGATE_ORDER = ["baseline", "no_uncorr"]

if abl_rgate is not None:
    abl_rgate_agg = aggregate(abl_rgate, ["variant", "noise"])
    display(abl_rgate_agg.sort_values(["noise", "variant"]))
    plot_ablation(abl_rgate_agg, B3F_RGATE_ORDER, title_prefix="BACON.3F Uncorrelatedness check")

## 3 BACON.7F ablations

In [ ]:
abl7 = load_csvs([RESULTS_DIR / "ablation_bacon7f.csv"])
B7F_ABL_ORDER = ["baseline", "no_relax", "no_folds"]

In [ ]:
if abl7 is not None:
    abl7_agg = aggregate(abl7, ["variant", "noise"])
    display(abl7_agg.sort_values(["noise", "variant"]))
    plot_ablation(abl7_agg, B7F_ABL_ORDER, title_prefix="BACON.7F",)

## 4 BACON.3F tuning

Default for each parameter is shown as a faint dotted grey vertical line.

In [ ]:
b3f = load_csvs([RESULTS_DIR / "parameters_bacon3f.csv"])

B3F_PARAM_MAP = {
    "md": ("max_depth", int),
    "ct": ("constancy_threshold", float),
    "rt": ("r_threshold", float),
    "r2": ("r2_threshold", float),
}
B3F_BASELINE = {
    "max_depth": 6,
    "constancy_threshold": 0.1,
    "r_threshold": 0.45,
    "r2_threshold": 0.9,
}

In [ ]:
if b3f is not None:
    b3f_curves = parse_variants(
        b3f,
        regex=r"^b3f_(md|ct|rt|r2)_([\d.]+)$",
        param_map=B3F_PARAM_MAP,
        baseline_variant="b3f_baseline",
        baseline_params=B3F_BASELINE,
    )
    b3f_agg = aggregate(
        b3f_curves.dropna(subset=["param"]),
        ["param", "param_value", "noise"],
    )

    plot_sensitivity_grid(
        b3f_agg, B3F_PARAM_MAP,
        title_prefix="BACON.3F",
        default_values=B3F_BASELINE,
    )

In [ ]:
if b3f is not None:
    print("(param, value) cells by success rate, mean R², and mean time at noise=0.0:")
    display(
        b3f_agg[b3f_agg["noise"] == 0.0]
        .sort_values(by=["success_rate", "r2_mean", "time_mean"], ascending=[False, False, True])
    )

In [ ]:
if b3f is not None:
    print("(param, value) cells by success rate, mean R², and mean time at noise=0.025:")
    display(
        b3f_agg[b3f_agg["noise"] == 0.025]
        .sort_values(by=["success_rate", "r2_mean", "time_mean"], ascending=[False, False, True])
    )

## 5 BACON.7F tuning

In [ ]:
b7f = load_csvs([RESULTS_DIR / "parameters_bacon7f.csv"])

B7F_PARAM_MAP = {
    "eps": ("initial_epsilon", float),
    "del": ("initial_delta", float),
    "cval": ("c_val", float),
    "sf": ("scale_factor", float),
    "nf": ("n_folds", int),
}
B7F_BASELINE = {
    "initial_epsilon": 0.01,
    "initial_delta": 0.1,
    "c_val": 0.2,
    "scale_factor": 1.2,
    "n_folds": 5,
}

In [ ]:
if b7f is not None:
    b7f_curves = parse_variants(
        b7f,
        regex=r"^b7f_(eps|del|cval|sf|nf)_([\d.]+)$",
        param_map=B7F_PARAM_MAP,
        baseline_variant="b7f_baseline",
        baseline_params=B7F_BASELINE,
    )
    b7f_agg = aggregate(
        b7f_curves.dropna(subset=["param"]),
        ["param", "param_value", "noise"],
    )

    plot_sensitivity_grid(
        b7f_agg, B7F_PARAM_MAP,
        title_prefix="BACON.7F",
        default_values=B7F_BASELINE,
    )

In [ ]:
'''if b7f is not None:
    print("(param, value) cells by success rate, mean R², and mean time at noise=0.0:")
    display(
        b7f_agg[b7f_agg["noise"] == 0.0]
        .sort_values(by=["success_rate", "r2_mean", "time_mean"],
                     ascending=[False, False, True])
    )'''

In [ ]:
'''if b7f is not None:
    print("(param, value) cells by success rate, mean R², and mean time at noise=0.025:")
    display(
        b7f_agg[b7f_agg["noise"] == 0.025]
        .sort_values(by=["success_rate", "r2_mean", "time_mean"],
                     ascending=[False, False, True])
    )'''

## 6 Comparison: BACON.3F vs BACON.7F vs PySR vs PySR (normalised)

In [ ]:
from symbolic_discovery.utils.analysis import equivalence_ratio

comparison = load_csvs([RESULTS_DIR / "comparison.csv"])

COMPARISON_ORDER = ["bacon3f", "bacon7f", "pysr", "pysr_norm"]
NOISE_LEVELS_FULL = [0.0, 0.001, 0.01, 0.025, 0.05]
N_SAMPLES_FULL = [10, 50, 100, 1000, 10000]

PALETTE_VARIANT = {
    "bacon3f":   "#298c8c",
    "bacon7f":   "#f1a226",
    "pysr":      "#3D5A80",
    "pysr_norm": "#C74E5B",
}

_ERR_COLS["std"]["sym_ratio_mean"] = "sym_ratio_std"
_ERR_COLS["sem"]["sym_ratio_mean"] = "sym_ratio_sem"


In [ ]:
def get_equivalence_ratio(equation: str, dataset: str) -> float:
    try:
        return equivalence_ratio(equation, dataset)
    except Exception:
        return float("nan")

def add_sym_ratio(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eqs = out["equation"].astype(str)
    out["sym_ratio"] = [
        get_equivalence_ratio(eq, ds) if found else 0.0
        for eq, ds, found in zip(eqs, out["dataset"], out["found"])
    ]
    return out

def aggregate_with_sym(df: pd.DataFrame, keys) -> pd.DataFrame:
    base = aggregate(df, keys)
    g = df.groupby(keys, dropna=False)["sym_ratio"]
    extras = pd.concat([
        g.mean().rename("sym_ratio_mean"),
        g.std().rename("sym_ratio_std"),
        g.sem().rename("sym_ratio_sem"),
    ], axis=1).reset_index()
    return base.merge(extras, on=list(keys), how="left")

if comparison is not None:
    comparison = add_sym_ratio(comparison)
    print(f"sym_ratio computed: "
          f"{(comparison['sym_ratio'] > 0).sum()}/{len(comparison)} "
          f"rows symbolically proportional to ground truth.")


### 6.1 Full aggregated table

In [ ]:
if comparison is not None:
    comparison_agg = aggregate_with_sym(
        comparison, ["variant", "noise", "n_samples"]
    )
    comparison_agg["variant"] = pd.Categorical(
        comparison_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    comparison_agg = comparison_agg.sort_values(
        ["variant", "n_samples", "noise"]
    ).reset_index(drop=True)
    display(comparison_agg)

### 6.2 Heatmaps

In [ ]:
NAMES = {"n_samples": "Sample size", "noise": "Noise level"}

def _annot_color(value: float, vmin: float, vmax: float, cmap_name: str) -> str:
    if not np.isfinite(value):
        return "#444444"
    cmap = matplotlib.colormaps[cmap_name]
    span = max(vmax - vmin, 1e-12)
    norm = (value - vmin) / span
    r, g, b, _ = cmap(np.clip(norm, 0.0, 1.0))
    lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return "white" if lum < 0.5 else "black"

def plot_comparison_heatmaps(
    agg: pd.DataFrame, metric: str, *,
    title: str,
    fmt: str = "{:.2f}",
    cmap: str = "viridis",
    vmin: float | None = None,
    vmax: float | None = None,
    savename: str | None = None,
) -> None:
    pivots: dict[str, pd.DataFrame] = {}
    for v in COMPARISON_ORDER:
        sub = agg[agg["variant"] == v]
        if sub.empty:
            pivots[v] = pd.DataFrame(
                np.nan, index=N_SAMPLES_FULL, columns=NOISE_LEVELS_FULL
            )
        else:
            pivots[v] = (
                sub.pivot(index="n_samples", columns="noise", values=metric)
                .reindex(index=N_SAMPLES_FULL, columns=NOISE_LEVELS_FULL)
            )

    if vmin is None or vmax is None:
        all_vals = np.concatenate([p.values.ravel() for p in pivots.values()])
        finite = all_vals[np.isfinite(all_vals)]
        auto_min = float(finite.min()) if finite.size else 0.0
        auto_max = float(finite.max()) if finite.size else 1.0
        vmin = auto_min if vmin is None else vmin
        vmax = auto_max if vmax is None else vmax
        if vmax - vmin < 1e-9:
            vmax = vmin + 1e-9

    fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)

    im = None
    for ax, variant in zip(axes.flat, COMPARISON_ORDER):
        data = pivots[variant].values
        masked = np.ma.masked_invalid(data)
        im = ax.imshow(
            masked, aspect="auto", cmap=cmap,
            vmin=vmin, vmax=vmax, origin="lower",
        )
        ax.set_xticks(range(len(NOISE_LEVELS_FULL)))
        ax.set_xticklabels([str(n) for n in NOISE_LEVELS_FULL])
        ax.set_yticks(range(len(N_SAMPLES_FULL)))
        ax.set_yticklabels([str(n) for n in N_SAMPLES_FULL])
        ax.set_xlabel(NAMES["noise"])
        ax.set_ylabel(NAMES["n_samples"])
        ax.set_title(PRETTY.get(variant, variant), loc="left", pad=8)
        ax.grid(False)
        for i in range(len(N_SAMPLES_FULL)):
            for j in range(len(NOISE_LEVELS_FULL)):
                v = data[i, j]
                if np.isfinite(v):
                    ax.text(
                        j, i, fmt.format(v),
                        ha="center", va="center",
                        color=_annot_color(v, vmin, vmax, cmap),
                    )
                else:
                    ax.text(j, i, "—", ha="center", va="center", color="#888")

    if im is not None:
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
    # fig.suptitle(title, y=1.03)
    if savename:
        fig.savefig(Path(f"figures/{savename}"))
    plt.show()


In [ ]:
if comparison is not None:
    plot_comparison_heatmaps(
        comparison_agg, "success_rate",
        title="Success rate",
        fmt="{:.2f}", cmap="viridis", vmin=0.0, vmax=1.0,
        savename="comparison_heatmap_success_rate.png",
    )


In [ ]:
if comparison is not None:
    plot_comparison_heatmaps(
        comparison_agg, "r2_mean",
        title=r"Mean $R^2$ (successful runs only)",
        fmt="{:.3f}", cmap="viridis", vmin=0.0, vmax=1.0,
        savename="comparison_heatmap_r2_mean.png",
    )


In [ ]:
if comparison is not None:
    plot_comparison_heatmaps(
        comparison_agg, "time_mean",
        title="Mean wall-clock time (s, all runs incl. failures)",
        fmt="{:.2f}", cmap="viridis_r",
        savename="comparison_heatmap_time_mean.png",
    )


In [ ]:
if comparison is not None:
    plot_comparison_heatmaps(
        comparison_agg, "time_mean_found",
        title="Mean wall-clock time on successful runs only (s)",
        fmt="{:.2f}", cmap="viridis_r",
        savename="comparison_heatmap_time_mean_found.png",
    )


In [ ]:
if comparison is not None:
    plot_comparison_heatmaps(
        comparison_agg, "sym_ratio_mean",
        title="Mean symbolic-equivalence ratio (0 = not proportional)",
        fmt="{:.2f}", cmap="viridis", vmin=0.0, vmax=1.0,
        savename="comparison_heatmap_sym_ratio.png",
    )


### 7.3 Bar charts

AGGREGATE OVER LOW SAMPLE SIZES

In [ ]:
if comparison is not None:
    LOW_N = [10, 50]
    LOWN_METRICS = [
        ("success_rate",         "Success rate"),
        ("r2_mean",              r"Mean $R^2$"),
        ("sym_ratio_mean_found", "Mean Sym. ratio"),
    ]

    lown = comparison[comparison["n_samples"].isin(LOW_N)]
    lown_agg = aggregate_with_sym(lown, ["variant", "noise"])

    # Add sym_ratio aggregates over successful runs only.
    g_found = (
        lown[lown["found"]]
        .groupby(["variant", "noise"], dropna=False)["sym_ratio"]
    )
    sym_found = pd.concat([
        g_found.mean().rename("sym_ratio_mean_found"),
        g_found.std().rename("sym_ratio_std_found"),
        g_found.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    lown_agg = lown_agg.merge(sym_found, on=["variant", "noise"], how="left")

    _ERR_COLS["std"]["sym_ratio_mean_found"] = "sym_ratio_std_found"
    _ERR_COLS["sem"]["sym_ratio_mean_found"] = "sym_ratio_sem_found"

    lown_agg["variant"] = pd.Categorical(
        lown_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )

    x = np.arange(len(COMPARISON_ORDER))
    bar_colors = [PALETTE_VARIANT.get(m, "#C74E5B") for m in COMPARISON_ORDER]
    ncols = len(LOWN_METRICS)
    ymargin = 0.12

    '''for no in NOISE_LEVELS_FULL:
        fig, axes = plt.subplots(
            1, ncols,
            figsize=(5.5 * ncols, 3.5),
            constrained_layout=True,
        )
        if ncols == 1:
            axes = np.array([axes])

        sub = (
            lown_agg[lown_agg["noise"] == no]
            .set_index("variant")
            .reindex(COMPARISON_ORDER)
        )
        for c, (col, ylabel) in enumerate(LOWN_METRICS):
            ax = axes[c]
            vals = sub[col].values
            yerr = _yerr(sub, col)
            bars = ax.bar(
                x, vals, 0.7,
                yerr=yerr, capsize=3,
                error_kw={"linewidth": 1.4, "ecolor": "#333"},
                color=bar_colors, edgecolor="none",
            )
            ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
            ax.set_xticks(x)
            ax.set_xticklabels(
                [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
            )

            err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
            lo_arr = np.asarray(vals, float) - err
            hi_arr = np.asarray(vals, float) + err
            finite_lo = lo_arr[np.isfinite(lo_arr)]
            finite_hi = hi_arr[np.isfinite(hi_arr)]
            if finite_lo.size and finite_hi.size:
                lo, hi = float(finite_lo.min()), float(finite_hi.max())
                if hi - lo < 1e-12:
                    pad = max(abs(hi) * 0.1, 0.01)
                    ax.set_ylim(lo - pad, hi + pad)
                else:
                    span = hi - lo
                    ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

            ax.set_title(ylabel, loc="left", pad=6)

        axes[0].set_ylabel(
            f"Noise={no}", rotation=90,
            va="center", labelpad=16, fontsize=18,
        )

        fig.savefig(Path(f"figures/comparisons/comparison_bars_lowN_noise{no}.png"))
        plt.show()'''
    
    # Extra: aggregated over all noise levels too. Only one used in dissertation.
    lown_all = aggregate_with_sym(lown, ["variant"])
    g_found_all = lown[lown["found"]].groupby("variant", dropna=False)["sym_ratio"]
    sym_found_all = pd.concat([
        g_found_all.mean().rename("sym_ratio_mean_found"),
        g_found_all.std().rename("sym_ratio_std_found"),
        g_found_all.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    lown_all = lown_all.merge(sym_found_all, on="variant", how="left")
    lown_all["variant"] = pd.Categorical(
        lown_all["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    sub = lown_all.set_index("variant").reindex(COMPARISON_ORDER)

    fig, axes = plt.subplots(
        1, ncols, figsize=(5 * ncols, 4), constrained_layout=True,
    )
    if ncols == 1:
        axes = np.array([axes])

    for c, (col, ylabel) in enumerate(LOWN_METRICS):
        ax = axes[c]
        vals = sub[col].values
        yerr = _yerr(sub, col)
        bars = ax.bar(
            x, vals, 0.7,
            yerr=yerr, capsize=3,
            error_kw={"linewidth": 1.4, "ecolor": "#333"},
            color=bar_colors, edgecolor="none",
        )
        ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
        ax.set_xticks(x)
        ax.set_xticklabels(
            [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
        )

        err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
        lo_arr = np.asarray(vals, float) - err
        hi_arr = np.asarray(vals, float) + err
        finite_lo = lo_arr[np.isfinite(lo_arr)]
        finite_hi = hi_arr[np.isfinite(hi_arr)]
        if finite_lo.size and finite_hi.size:
            lo, hi = float(finite_lo.min()), float(finite_hi.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi) * 0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

        ax.set_title(ylabel, loc="left", pad=6)

    '''axes[0].set_ylabel(
        "All noise", rotation=90,
        va="center", labelpad=16, fontsize=18,
    )'''

    fig.savefig(Path("figures/comparisons/comparison_bars_lowN_allnoise.png"))
    plt.show()

    '''# Extra extra: aggregated over noise levels excluding noise=0.05.
    lown_no005 = lown[lown["noise"] != 0.05]
    lown_no005_agg = aggregate_with_sym(lown_no005, ["variant"])
    g_found_no005 = lown_no005[lown_no005["found"]].groupby("variant", dropna=False)["sym_ratio"]
    sym_found_no005 = pd.concat([
        g_found_no005.mean().rename("sym_ratio_mean_found"),
        g_found_no005.std().rename("sym_ratio_std_found"),
        g_found_no005.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    lown_no005_agg = lown_no005_agg.merge(sym_found_no005, on="variant", how="left")
    lown_no005_agg["variant"] = pd.Categorical(
        lown_no005_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    sub = lown_no005_agg.set_index("variant").reindex(COMPARISON_ORDER)

    fig, axes = plt.subplots(
        1, ncols, figsize=(5 * ncols, 4), constrained_layout=True,
    )
    if ncols == 1:
        axes = np.array([axes])

    for c, (col, ylabel) in enumerate(LOWN_METRICS):
        ax = axes[c]
        vals = sub[col].values
        yerr = _yerr(sub, col)
        bars = ax.bar(
            x, vals, 0.7,
            yerr=yerr, capsize=3,
            error_kw={"linewidth": 1.4, "ecolor": "#333"},
            color=bar_colors, edgecolor="none",
        )
        ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
        ax.set_xticks(x)
        ax.set_xticklabels(
            [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
        )

        err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
        lo_arr = np.asarray(vals, float) - err
        hi_arr = np.asarray(vals, float) + err
        finite_lo = lo_arr[np.isfinite(lo_arr)]
        finite_hi = hi_arr[np.isfinite(hi_arr)]
        if finite_lo.size and finite_hi.size:
            lo, hi = float(finite_lo.min()), float(finite_hi.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi) * 0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

        ax.set_title(ylabel, loc="left", pad=6)

    axes[0].set_ylabel(
        "Noise≤0.025", rotation=90,
        va="center", labelpad=16, fontsize=18,
    )

    fig.savefig(Path("figures/comparisons/comparison_bars_lowN_allnoise_no005.png"))
    plt.show()'''

AGGREGATE OVER HIGH NOISES

In [ ]:
if comparison is not None:
    HIGH_NOISE = [0.025, 0.05]
    HIGHN_METRICS = [
        ("success_rate", "Success rate"),
        ("r2_mean", r"Mean $R^2$"),
        ("sym_ratio_mean_found", "Mean Sym. ratio"),
    ]

    highn = comparison[comparison["noise"].isin(HIGH_NOISE)]
    highn_agg = aggregate_with_sym(highn, ["variant", "n_samples"])

    # Add sym_ratio aggregates over successful runs only.
    g_found = (
        highn[highn["found"]]
        .groupby(["variant", "n_samples"], dropna=False)["sym_ratio"]
    )
    sym_found = pd.concat([
        g_found.mean().rename("sym_ratio_mean_found"),
        g_found.std().rename("sym_ratio_std_found"),
        g_found.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    highn_agg = highn_agg.merge(sym_found, on=["variant", "n_samples"], how="left")

    _ERR_COLS["std"]["sym_ratio_mean_found"] = "sym_ratio_std_found"
    _ERR_COLS["sem"]["sym_ratio_mean_found"] = "sym_ratio_sem_found"

    highn_agg["variant"] = pd.Categorical(
        highn_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )

    x = np.arange(len(COMPARISON_ORDER))
    bar_colors = [PALETTE_VARIANT.get(m, "#666666") for m in COMPARISON_ORDER]
    ncols = len(HIGHN_METRICS)
    ymargin = 0.12

    '''for ns in N_SAMPLES_FULL:
        fig, axes = plt.subplots(
            1, ncols,
            figsize=(5.5 * ncols, 3.5),
            constrained_layout=True,
        )
        if ncols == 1:
            axes = np.array([axes])

        sub = (
            highn_agg[highn_agg["n_samples"] == ns]
            .set_index("variant")
            .reindex(COMPARISON_ORDER)
        )
        for c, (col, ylabel) in enumerate(HIGHN_METRICS):
            ax = axes[c]
            vals = sub[col].values
            yerr = _yerr(sub, col)
            bars = ax.bar(
                x, vals, 0.7,
                yerr=yerr, capsize=3,
                error_kw={"linewidth": 1.4, "ecolor": "#333"},
                color=bar_colors, edgecolor="none",
            )
            ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
            ax.set_xticks(x)
            ax.set_xticklabels(
                [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
            )

            err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
            lo_arr = np.asarray(vals, float) - err
            hi_arr = np.asarray(vals, float) + err
            finite_lo = lo_arr[np.isfinite(lo_arr)]
            finite_hi = hi_arr[np.isfinite(hi_arr)]
            if finite_lo.size and finite_hi.size:
                lo, hi = float(finite_lo.min()), float(finite_hi.max())
                if hi - lo < 1e-12:
                    pad = max(abs(hi) * 0.1, 0.01)
                    ax.set_ylim(lo - pad, hi + pad)
                else:
                    span = hi - lo
                    ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

            ax.set_title(ylabel, loc="left", pad=6)

        axes[0].set_ylabel(
            f"N={ns}", rotation=90,
            va="center", labelpad=16, fontsize=18,
        )

        fig.savefig(Path(f"figures/comparisons/comparison_bars_highNoise_n{ns}.png"))
        plt.show()'''

    # Extra: aggregated over all sample sizes too.
    highn_all = aggregate_with_sym(highn, ["variant"])
    g_found_all = highn[highn["found"]].groupby("variant", dropna=False)["sym_ratio"]
    sym_found_all = pd.concat([
        g_found_all.mean().rename("sym_ratio_mean_found"),
        g_found_all.std().rename("sym_ratio_std_found"),
        g_found_all.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    highn_all = highn_all.merge(sym_found_all, on="variant", how="left")
    highn_all["variant"] = pd.Categorical(
        highn_all["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    sub = highn_all.set_index("variant").reindex(COMPARISON_ORDER)

    fig, axes = plt.subplots(
        1, ncols, figsize=(5 * ncols, 4), constrained_layout=True,
    )
    if ncols == 1:
        axes = np.array([axes])

    for c, (col, ylabel) in enumerate(HIGHN_METRICS):
        ax = axes[c]
        vals = sub[col].values
        yerr = _yerr(sub, col)
        bars = ax.bar(
            x, vals, 0.7,
            yerr=yerr, capsize=3,
            error_kw={"linewidth": 1.4, "ecolor": "#333"},
            color=bar_colors, edgecolor="none",
        )
        ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
        ax.set_xticks(x)
        ax.set_xticklabels(
            [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
        )

        err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
        lo_arr = np.asarray(vals, float) - err
        hi_arr = np.asarray(vals, float) + err
        finite_lo = lo_arr[np.isfinite(lo_arr)]
        finite_hi = hi_arr[np.isfinite(hi_arr)]
        if finite_lo.size and finite_hi.size:
            lo, hi = float(finite_lo.min()), float(finite_hi.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi) * 0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

        ax.set_title(ylabel, loc="left", pad=6)

    '''axes[0].set_ylabel(
        "All N", rotation=90,
        va="center", labelpad=16, fontsize=18,
    )'''

    fig.savefig(Path("figures/comparison_bars_highNoise_allN.png"))
    plt.show()

    # Extra extra: aggregated over sample sizes excluding n=10.
    highn_no10 = highn[highn["n_samples"] != 10]
    highn_no10_agg = aggregate_with_sym(highn_no10, ["variant"])
    g_found_no10 = highn_no10[highn_no10["found"]].groupby("variant", dropna=False)["sym_ratio"]
    sym_found_no10 = pd.concat([
        g_found_no10.mean().rename("sym_ratio_mean_found"),
        g_found_no10.std().rename("sym_ratio_std_found"),
        g_found_no10.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    highn_no10_agg = highn_no10_agg.merge(sym_found_no10, on="variant", how="left")
    highn_no10_agg["variant"] = pd.Categorical(
        highn_no10_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    sub = highn_no10_agg.set_index("variant").reindex(COMPARISON_ORDER)

    fig, axes = plt.subplots(
        1, ncols, figsize=(5.5 * ncols, 3.5), constrained_layout=True,
    )
    if ncols == 1:
        axes = np.array([axes])

    for c, (col, ylabel) in enumerate(HIGHN_METRICS):
        ax = axes[c]
        vals = sub[col].values
        yerr = _yerr(sub, col)
        bars = ax.bar(
            x, vals, 0.7,
            yerr=yerr, capsize=3,
            error_kw={"linewidth": 1.4, "ecolor": "#333"},
            color=bar_colors, edgecolor="none",
        )
        ax.bar_label(bars, fmt="%.3f", padding=2, color="#444")
        ax.set_xticks(x)
        ax.set_xticklabels(
            [PRETTY.get(m, m) for m in COMPARISON_ORDER], fontsize=14
        )

        err = np.asarray(yerr if yerr is not None else np.zeros_like(vals), float)
        lo_arr = np.asarray(vals, float) - err
        hi_arr = np.asarray(vals, float) + err
        finite_lo = lo_arr[np.isfinite(lo_arr)]
        finite_hi = hi_arr[np.isfinite(hi_arr)]
        if finite_lo.size and finite_hi.size:
            lo, hi = float(finite_lo.min()), float(finite_hi.max())
            if hi - lo < 1e-12:
                pad = max(abs(hi) * 0.1, 0.01)
                ax.set_ylim(lo - pad, hi + pad)
            else:
                span = hi - lo
                ax.set_ylim(lo - span * ymargin, hi + span * (ymargin + 0.15))

        ax.set_title(ylabel, loc="left", pad=6)

    axes[0].set_ylabel(
        "N≥50", rotation=90,
        va="center", labelpad=16, fontsize=18,
    )

    fig.savefig(Path("figures/comparison_bars_highNoise_allN_no10.png"))
    plt.show()

AVERAGE OVER ALL

In [ ]:
if comparison is not None:
    sub = comparison[comparison["n_samples"] != 100000]

    overall_agg = aggregate_with_sym(sub, ["variant"])

    # sym_ratio aggregates over successful runs only
    g_found = sub[sub["found"]].groupby("variant", dropna=False)["sym_ratio"]
    sym_found = pd.concat([
        g_found.mean().rename("sym_ratio_mean_found"),
        g_found.std().rename("sym_ratio_std_found"),
        g_found.sem().rename("sym_ratio_sem_found"),
    ], axis=1).reset_index()
    overall_agg = overall_agg.merge(sym_found, on="variant", how="left")

    overall_agg["variant"] = pd.Categorical(
        overall_agg["variant"], categories=COMPARISON_ORDER, ordered=True
    )
    overall_agg = overall_agg.sort_values("variant").reset_index(drop=True)
    display(overall_agg)

AGGREGATE OVER NOISE AND SAMPLE SIZE SEPARATELY

In [ ]:
if comparison is not None:
    from matplotlib.ticker import MaxNLocator

    LINE_METRICS = [
        ("success_rate",         None,                  "Success rate"),
        ("r2_mean",              "r2_sem",              r"Mean $R^2$"),
        ("sym_ratio_mean_found", "sym_ratio_sem_found", "Mean Sym. ratio"),
    ]

    def _aggregate_for_lines(raw, group_keys):
        agg = aggregate_with_sym(raw, group_keys)
        g = raw[raw["found"]].groupby(group_keys, dropna=False)["sym_ratio"]
        extras = pd.concat([
            g.mean().rename("sym_ratio_mean_found"),
            g.std().rename("sym_ratio_std_found"),
            g.sem().rename("sym_ratio_sem_found"),
        ], axis=1).reset_index()
        return agg.merge(extras, on=list(group_keys), how="left")

    agg_by_n     = _aggregate_for_lines(comparison, ["variant", "n_samples"])
    agg_by_noise = _aggregate_for_lines(comparison, ["variant", "noise"])

    fig, axes = plt.subplots(
        len(LINE_METRICS), 2, figsize=(11, 3.2 * len(LINE_METRICS)),
        constrained_layout=True,
    )

    def _draw(ax, agg, x_col, x_values, mean_col, sem_col, *, log_x=False):
        rows = {}
        for v in COMPARISON_ORDER:
            sub = (
                agg[agg["variant"] == v]
                .set_index(x_col)
                .reindex(x_values)
                .reset_index()
            )
            xs = sub[x_col].values if log_x else np.arange(len(x_values))
            ys = sub[mean_col].values
            if sem_col is not None and sem_col in sub.columns:
                errs = sub[sem_col].fillna(0).values
            else:
                errs = np.zeros_like(ys)
            color = PALETTE_VARIANT.get(v, "#666666")
            ax.plot(xs, ys, "-o", color=color, label=PRETTY.get(v, v),
                    linewidth=2, markersize=5)
            if np.any(errs > 0):
                ax.fill_between(
                    xs, ys - errs, ys + errs,
                    color=color, alpha=0.18, linewidth=0,
                )
            rows[v] = ys
        if log_x:
            ax.set_xscale("log")
            ax.set_xticks(x_values)
            ax.set_xticklabels([str(x) for x in x_values])
        else:
            ax.set_xticks(np.arange(len(x_values)))
            ax.set_xticklabels([str(x) for x in x_values])
        ax.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 2.5, 5, 10]))
        return rows

    all_tables = []
    for r, (mean_col, sem_col, ylabel) in enumerate(LINE_METRICS):
        rows_left = _draw(axes[r, 0], agg_by_n, "n_samples", N_SAMPLES_FULL,
                          mean_col, sem_col, log_x=True)
        rows_right = _draw(axes[r, 1], agg_by_noise, "noise", NOISE_LEVELS_FULL,
                           mean_col, sem_col, log_x=False)

        df_left = pd.DataFrame(
            {PRETTY.get(v, v): rows_left[v] for v in COMPARISON_ORDER},
            index=N_SAMPLES_FULL,
        )
        df_left.index.name = "n_samples"
        df_right = pd.DataFrame(
            {PRETTY.get(v, v): rows_right[v] for v in COMPARISON_ORDER},
            index=NOISE_LEVELS_FULL,
        )
        df_right.index.name = "noise"

        all_tables.append((ylabel, df_left, df_right))

        axes[r, 0].set_ylabel(ylabel)
        if r == len(LINE_METRICS) - 1:
            axes[r, 0].set_xlabel("Sample size")
            axes[r, 1].set_xlabel("Noise level")
        if r == 0:
            axes[r, 0].set_title("Aggregated over noise", loc="left", pad=6)
            axes[r, 1].set_title("Aggregated over sample size", loc="left", pad=6)

    # axes[1, 0].set_ylim(bottom=-1.0, top=1.05)
    # axes[1, 1].set_ylim(bottom=-1.0, top=1.05)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center", bbox_to_anchor=(0.5, 1.04),
        ncol=len(COMPARISON_ORDER), frameon=False, fontsize=14,
    )

    fig.savefig(Path("figures/comparison_lines_2x2.png"))
    plt.show()

    for ylabel, df_left, df_right in all_tables:
        print(f"\n=== {ylabel} ===")
        print("\n  vs sample size (aggregated over noise):")
        print(df_left.round(3).to_string())
        print("\n  vs noise (aggregated over sample size):")
        print(df_right.round(3).to_string())